# Chapter 2 — Solutions of Equations in One Variable

## 2.1 The Bisection Method

Given a continuous function $f$ on $[a,b]$ with $f(a)$ and $f(b)$ of opposite sign, the **Intermediate Value Theorem** guarantees at least one root $p$ in $(a,b)$.

The bisection method repeatedly halves the interval, choosing the sub-interval where the sign change occurs.

**Error bound after $n$ steps:**

$$|p_n - p| \le \frac{b-a}{2^n}$$

In [1]:
import numpy as np
import pandas as pd

In [2]:
def bisection(f, a, b, tol=1e-10, max_iter=100):
    """
    Bisection method for finding a root of f on [a, b].

    Returns (p, steps) where p is the approximate root and steps is a list
    of dicts recording each iteration.
    """
    if f(a) * f(b) >= 0:
        raise ValueError("f(a) and f(b) must have opposite signs")

    steps = []
    for n in range(1, max_iter + 1):
        p = a + (b - a) / 2          # midpoint (avoids overflow vs (a+b)/2)
        fp = f(p)
        steps.append({"n": n, "a": a, "b": b, "p": p, "f(p)": fp})

        if fp == 0 or (b - a) / 2 < tol:
            return p, steps

        if f(a) * fp < 0:
            b = p
        else:
            a = p

    raise RuntimeError(f"Method did not converge in {max_iter} iterations")

### Example — $f(x) = \cos(x) - x$ on $[0, 1]$

The exact root is the Dottie number: $p \approx 0.7390851332$.

In [6]:
f = lambda x: np.cos(x) - x

p_exact = 0.7390851332151607   # known root (Dottie number)

root, steps = bisection(f, 0, 1, tol=1e-10)

df = pd.DataFrame(steps)
df["|error|"] = np.abs(df["p"] - p_exact)
df["bound"]  = (1 - 0) / 2**df["n"]     # theoretical error bound (b-a)/2^n

pd.set_option("display.float_format", "{:.10f}".format)
df

,n,a,b,p,f(p),|error|,bound
0,1,0.0000000000,1.0000000000,0.5000000000,0.3775825619,0.2390851332,0.5000000000
1,2,0.5000000000,1.0000000000,0.7500000000,-0.0183111311,0.0109148668,0.2500000000
2,3,0.5000000000,0.7500000000,0.6250000000,0.1859631195,0.1140851332,0.1250000000
3,4,0.6250000000,0.7500000000,0.6875000000,0.0853349462,0.0515851332,0.0625000000
4,5,0.6875000000,0.7500000000,0.7187500000,0.0338793724,0.0203351332,0.0312500000
5,6,0.7187500000,0.7500000000,0.7343750000,0.0078747255,0.0047101332,0.0156250000
6,7,0.7343750000,0.7500000000,0.7421875000,-0.0051957117,0.0031023668,0.0078125000
7,8,0.7343750000,0.7421875000,0.7382812500,0.0013451498,0.0008038832,0.0039062500
8,9,0.7382812500,0.7421875000,0.7402343750,-0.0019238728,0.0011492418,0.0019531250
9,10,0.7382812500,0.7402343750,0.7392578125,-0.0002890091,0.0001726793,0.0009765625


## 2.2 Fixed-Point Iteration

A **fixed point** of $g$ is a number $p$ such that $g(p)=p$.

If $p$ is a root of $f$, i.e. $f(p)=0$, then for any function of the form $g(x)=x + c\,f(x)$ we have $g(p)=p$.

The fixed-point iteration simply repeats $p_n = g(p_{n-1})$.

**Convergence condition:** if $|g'(x)| < 1$ for all $x$ in a neighbourhood of $p$, the iteration converges.

We test two choices for $f(x)=\cos x - x$:
- $h_1(x) = x + f(x) = x + \cos x - x = \cos x \quad\Rightarrow\quad h_1'(x)=-\sin x$
  At $p\approx0.739$: $|h_1'(p)|=|\!-\!\sin(0.739)|\approx0.674 < 1$ ✓ — **converges**
- $h_2(x) = x - f(x) = x - \cos x + x = 2x - \cos x \quad\Rightarrow\quad h_2'(x)=2+\sin x$
  At $p\approx0.739$: $|h_2'(p)|=2+\sin(0.739)\approx2.674 > 1$ ✗ — **diverges**

In [7]:
def fixed_point(g, p0, tol=1e-10, max_iter=100):
    """
    Fixed-point iteration: p_n = g(p_{n-1}).

    Returns (p, steps) where steps logs each iteration.
    """
    steps = [{"n": 0, "p": p0}]
    for n in range(1, max_iter + 1):
        p = g(steps[-1]["p"])
        steps.append({"n": n, "p": p})
        if abs(p - steps[-2]["p"]) < tol:
            return p, steps
    raise RuntimeError(f"Method did not converge in {max_iter} iterations")

### $h_1(x) = \cos x$ — should converge

In [8]:
h1 = lambda x: np.cos(x)            # g(x) = x + f(x) = cos(x)

root1, steps1 = fixed_point(h1, 0.5, tol=1e-10)

df1 = pd.DataFrame(steps1)
df1["|error|"] = np.abs(df1["p"] - p_exact)
df1

,n,p,|error|
0,0,0.5000000000,0.2390851332
1,1,0.8775825619,0.1384974287
2,2,0.6390124942,0.1000726390
3,3,0.8026851007,0.0635999675
4,4,0.6947780268,0.0443071064
5,5,0.7681958313,0.0291106981
6,6,0.7191654459,0.0199196873
7,7,0.7523557594,0.0132706262
8,8,0.7300810631,0.0090040701
9,9,0.7451203414,0.0060352081


### $h_2(x) = 2x - \cos x$ — should diverge

In [9]:
h2 = lambda x: 2*x - np.cos(x)      # g(x) = x - f(x) = 2x - cos(x)

try:
    root2, steps2 = fixed_point(h2, 0.5, tol=1e-10, max_iter=30)
except RuntimeError as e:
    # collect partial steps manually so we can see the divergence
    steps2 = [{"n": 0, "p": 0.5}]
    for n in range(1, 31):
        p = h2(steps2[-1]["p"])
        steps2.append({"n": n, "p": p})

df2 = pd.DataFrame(steps2)
df2["|error|"] = np.abs(df2["p"] - p_exact)
df2

,n,p,|error|
0,0,0.5000000000,0.2390851332
1,1,0.1224174381,0.6166676951
2,2,-0.7476814621,1.4867665953
3,3,-2.2286302303,2.9677153635
4,4,-3.8458562450,4.5849413783
5,5,-6.9296239272,7.6687090604
6,6,-14.6574818986,15.3965670318
7,7,-28.8178103579,29.5568954911
8,8,-56.7797045952,57.5187897284
9,9,-114.5328386880,115.2719238213


### $h_3(x) = x - \dfrac{f(x)}{f'(x)}$ (Newton's method)

Note: $f(x)=\cos x - x$, so $f'(x)=-\sin x - 1$.

This is exactly **Newton's method**. Since $f(p)=0$ we have $h_3(p)=p$, and
$h_3'(x) = \dfrac{f(x)\,f''(x)}{[f'(x)]^2}$, so $h_3'(p)=0$ — **quadratic convergence**.

In [11]:
f_prime = lambda x: -np.sin(x) - 1
h3 = lambda x: x - f(x) / f_prime(x)

try:
    root3, steps3 = fixed_point(h3, 0.5, tol=1e-10, max_iter=30)
except RuntimeError as e:
    steps3 = [{"n": 0, "p": 0.5}]
    for n in range(1, 31):
        p = h3(steps3[-1]["p"])
        steps3.append({"n": n, "p": p})

df3 = pd.DataFrame(steps3)
df3["|error|"] = np.abs(df3["p"] - p_exact)
df3

,n,p,|error|
0,0,0.5000000000,0.2390851332
1,1,0.7552224171,0.0161372839
2,2,0.7391416661,0.0000565329
3,3,0.7390851339,0.0000000007
4,4,0.7390851332,0.0000000000
5,5,0.7390851332,0.0000000000


### Example — $f(x) = (x-\pi)^3$ on $[3, 3.5]$

The exact root is $p=\pi$.  This is a **triple root** ($f'(\pi)=0$, $f''(\pi)=0$), so Newton's method loses its quadratic convergence and falls back to **linear convergence**.

In [12]:
g = lambda x: (x - np.pi)**3
g_prime = lambda x: 3*(x - np.pi)**2

p_exact_g = np.pi

# --- Bisection ---
root_bis, steps_bis = bisection(g, 3, 3.5, tol=1e-10)

df_bis = pd.DataFrame(steps_bis)
df_bis["|error|"] = np.abs(df_bis["p"] - p_exact_g)

# --- Newton's method via fixed-point iteration ---
h_newton = lambda x: x - g(x) / g_prime(x)

root_new, steps_new = fixed_point(h_newton, 3.0, tol=1e-10, max_iter=200)

df_new = pd.DataFrame(steps_new)
df_new["|error|"] = np.abs(df_new["p"] - p_exact_g)

print(f"Bisection:  {len(df_bis)} iterations")
print(f"Newton:     {len(df_new)} iterations")
print()

print("=== Bisection (last 10 steps) ===")
display(df_bis.tail(10))

print("\n=== Newton's method (last 10 steps) ===")
display(df_new.tail(10))

Bisection:  33 iterations
Newton:     52 iterations

=== Bisection (last 10 steps) ===


,n,a,b,p,f(p),|error|
23,24,3.1415926218,3.1415926814,3.1415926516,-0.0000000000,0.0000000020
24,25,3.1415926516,3.1415926814,3.1415926665,0.0000000000,0.0000000129
25,26,3.1415926516,3.1415926665,3.1415926591,0.0000000000,0.0000000055
26,27,3.1415926516,3.1415926591,3.1415926553,0.0000000000,0.0000000017
27,28,3.1415926516,3.1415926553,3.1415926535,-0.0000000000,0.0000000001
28,29,3.1415926535,3.1415926553,3.1415926544,0.0000000000,0.0000000008
29,30,3.1415926535,3.1415926544,3.1415926539,0.0000000000,0.0000000003
30,31,3.1415926535,3.1415926539,3.1415926537,0.0000000000,0.0000000001
31,32,3.1415926535,3.1415926537,3.1415926536,-0.0000000000,0.0000000000
32,33,3.1415926536,3.1415926537,3.1415926536,0.0000000000,0.0000000001



=== Newton's method (last 10 steps) ===


,n,p,|error|
42,42,3.1415926479,0.0000000057
43,43,3.1415926498,0.0000000038
44,44,3.1415926511,0.0000000025
45,45,3.1415926519,0.0000000017
46,46,3.1415926525,0.0000000011
47,47,3.1415926528,0.0000000007
48,48,3.1415926531,0.0000000005
49,49,3.1415926533,0.0000000003
50,50,3.1415926534,0.0000000002
51,51,3.1415926534,0.0000000001


## 2.3 Newton's Method, Secant Method, and False Position

All three methods for $f(x)=\cos x - x$ on $[0,1]$, with exact root $p\approx 0.7390851332$.

| Method | Formula | Convergence |
|--------|---------|-------------|
| **Newton** | $p_n = p_{n-1} - \dfrac{f(p_{n-1})}{f'(p_{n-1})}$ | Quadratic (requires $f'$) |
| **Secant** | $p_n = p_{n-1} - f(p_{n-1})\dfrac{p_{n-1}-p_{n-2}}{f(p_{n-1})-f(p_{n-2})}$ | Super-linear ($\approx 1.618$) |
| **False Position** | Like secant, but always keeps a bracket | Linear (but safe) |

In [ ]:
def newton(f, f_prime, p0, tol=1e-10, max_iter=100):
    """Newton's method."""
    steps = [{"n": 0, "p": p0}]
    for n in range(1, max_iter + 1):
        p = p0 - f(p0) / f_prime(p0)
        steps.append({"n": n, "p": p})
        if abs(p - p0) < tol:
            return p, steps
        p0 = p
    raise RuntimeError(f"Newton did not converge in {max_iter} iterations")


def secant(f, p0, p1, tol=1e-10, max_iter=100):
    """Secant method."""
    steps = [{"n": 0, "p": p0}, {"n": 1, "p": p1}]
    fp0 = f(p0)  # precompute f(p0) for first iteration
    for n in range(2, max_iter + 1):
        fp1 = f(p1)
        p = p1 - fp1 * (p1 - p0) / (fp1 - fp0)
        steps.append({"n": n, "p": p})
        if abs(p - p1) < tol:
            return p, steps
        p0, p1 = p1, p
        fp0 = fp1  # update fp0 for next iteration
    raise RuntimeError(f"Secant did not converge in {max_iter} iterations")


def false_position(f, a, b, tol=1e-10, max_iter=100):
    """Method of False Position (Regula Falsi)."""
    if f(a) * f(b) >= 0:
        raise ValueError("f(a) and f(b) must have opposite signs")
    steps = []
    for n in range(1, max_iter + 1):
        fa, fb = f(a), f(b)
        p = a - fa * (b - a) / (fb - fa)     # secant line zero
        fp = f(p)
        steps.append({"n": n, "a": a, "b": b, "p": p, "f(p)": fp})
        if abs(fp) < tol:
            return p, steps
        if fa * fp < 0:
            b = p
        else:
            a = p
    raise RuntimeError(f"False position did not converge in {max_iter} iterations")

In [14]:
f = lambda x: np.cos(x) - x
f_prime = lambda x: -np.sin(x) - 1
p_exact = 0.7390851332151607

# Newton's method  (starting at x=0.5)
root_n, steps_n = newton(f, f_prime, 0.5, tol=1e-10)
df_n = pd.DataFrame(steps_n)
df_n["|error|"] = np.abs(df_n["p"] - p_exact)

# Secant method  (starting at x=0, x=1)
root_s, steps_s = secant(f, 0, 1, tol=1e-10)
df_s = pd.DataFrame(steps_s)
df_s["|error|"] = np.abs(df_s["p"] - p_exact)

# False Position  (bracket [0, 1])
root_fp, steps_fp = false_position(f, 0, 1, tol=1e-10)
df_fp = pd.DataFrame(steps_fp)
df_fp["|error|"] = np.abs(df_fp["p"] - p_exact)

print(f"Newton:          {len(df_n)-1} iterations,  root = {root_n:.12f}")
print(f"Secant:          {len(df_s)-2} iterations,  root = {root_s:.12f}")
print(f"False Position:  {len(df_fp)} iterations,  root = {root_fp:.12f}")

print("\n=== Newton ===")
display(df_n)

print("\n=== Secant ===")
display(df_s)

print("\n=== False Position ===")
display(df_fp)

Newton:          5 iterations,  root = 0.739085133215
Secant:          6 iterations,  root = 0.739085133215
False Position:  8 iterations,  root = 0.739085133171

=== Newton ===


,n,p,|error|
0,0,0.5000000000,0.2390851332
1,1,0.7552224171,0.0161372839
2,2,0.7391416661,0.0000565329
3,3,0.7390851339,0.0000000007
4,4,0.7390851332,0.0000000000
5,5,0.7390851332,0.0000000000



=== Secant ===


,n,p,|error|
0,0,0.0000000000,0.7390851332
1,1,1.0000000000,0.2609148668
2,2,0.6850733573,0.0540117759
3,3,0.7362989976,0.0027861356
4,4,0.7391193619,0.0000342287
5,5,0.7390851121,0.0000000211
6,6,0.7390851332,0.0000000000
7,7,0.7390851332,0.0000000000



=== False Position ===


,n,a,b,p,f(p),|error|
0,1,0.0000000000,1,0.6850733573,0.0892992765,0.0540117759
1,2,0.6850733573,1,0.7362989976,0.0046600390,0.0027861356
2,3,0.7362989976,1,0.7389453560,0.0002339257,0.0001397772
3,4,0.7389453560,1,0.7390781309,0.0000117192,0.0000070023
4,5,0.7390781309,1,0.7390847824,0.0000005870,0.0000003508
5,6,0.7390847824,1,0.7390851156,0.0000000294,0.0000000176
6,7,0.7390851156,1,0.7390851323,0.0000000015,0.0000000009
7,8,0.7390851323,1,0.7390851332,0.0000000001,0.0000000000
